# 03 · `multiprocessing` in Depth: Processes, Pools, Communication & Shared Memory

**Module 13 – Concurrency & asyncio** · Estimated time: 60 min

### 🎯 What you will learn
- `Process` objects, exit codes, the `if __name__ == "__main__"` rule
- **Start methods**: `spawn`, `fork`, `forkserver` (and platform defaults)
- `Pool`: `map`, `imap`, `imap_unordered`, `starmap`, `apply_async`, `chunksize`, initialisers
- Communication: `Queue`, `Pipe`, `Event`, `Lock` across processes
- Sharing state: `Value`, `Array`, `shared_memory`, `Manager`
- Pickling constraints and performance tips

> ⚠️ With the `spawn` start method (default on macOS and Windows), child processes **import your module** to find the target function. Functions defined only inside a notebook can't be found, so we write them to a module file first (`%%writefile`). In scripts, always put process-starting code under `if __name__ == "__main__":`.

In [1]:
%%writefile mp_tasks.py
import os, time, math

def square(x):
    return x * x

def slow_square(x):
    time.sleep(0.05)
    return x * x

def who_am_i(tag):
    return f"{tag} ran in process {os.getpid()}"

def is_prime(n):
    if n < 2:
        return False
    return all(n % d for d in range(2, math.isqrt(n) + 1))

def power(base, exp):
    return base ** exp

def producer(q, n):
    for i in range(n):
        q.put(i)
    q.put(None)                       # sentinel: "no more items"

def consumer(q, results):
    while (item := q.get()) is not None:
        results.put(item * 10)

def ping_pong(conn):
    msg = conn.recv()
    conn.send(f"pong ({msg})")
    conn.close()

def add_to_counter(counter, lock, n):
    for _ in range(n):
        with lock:
            counter.value += 1

def fill_array(arr, offset):
    for i in range(len(arr)):
        arr[i] = i + offset

def init_worker(prefix):
    global PREFIX
    PREFIX = prefix

def tagged(x):
    return f"{PREFIX}-{x}"

def record(shared_dict, shared_list, key):
    shared_dict[key] = os.getpid()
    shared_list.append(key)

def double_shm(name, size):
    from multiprocessing import shared_memory
    import array
    shm = shared_memory.SharedMemory(name=name)
    view = shm.buf.cast("i")
    for i in range(size):
        view[i] *= 2
    view.release()
    shm.close()

Writing mp_tasks.py


In [2]:
import multiprocessing as mp, os, sys, time
import mp_tasks
print("parent pid:", os.getpid(), "| CPU cores:", os.cpu_count(), "| start method:", mp.get_start_method())

parent pid: 6277 | CPU cores: 8 | start method: spawn


## 1. `Process`

In [3]:
p = mp.Process(target=mp_tasks.slow_square, args=(7,), name="sq-worker")
p.start()
print(p.name, "alive:", p.is_alive(), "pid:", p.pid)
p.join(timeout=5)
print("exit code:", p.exitcode)                 # 0 = success; negative = killed by signal

sq-worker alive: True pid: 6281
exit code: 0


A `Process` can't *return* a value to the parent; use a `Pool`, a `Queue`, or `concurrent.futures` (lesson 04).

## 2. Start methods

| Method | How the child starts | Default on |
|---|---|---|
| `spawn` | a fresh interpreter imports your module | **macOS** (3.8+), **Windows** |
| `fork` | copies the parent process (fast; unsafe with threads) | older Linux defaults |
| `forkserver` | a clean server process forks children on request | 🆕 **Linux since 3.14** |

```python
ctx = mp.get_context("spawn")       # choose explicitly for portable code
with ctx.Pool(4) as pool: ...
```
> ⚠️ `fork` + threads can deadlock (a lock held by another thread is copied in the locked state). Python warns about forking multi-threaded processes; prefer `spawn`/`forkserver`.

## 3. `Pool`: the workhorse

In [4]:
with mp.Pool(processes=4) as pool:
    print(pool.map(mp_tasks.square, range(10)))                        # ordered results, blocks
    print(pool.starmap(mp_tasks.power, [(2, 3), (3, 2), (10, 3)]))     # multiple arguments
    print(sorted(set(pool.map(mp_tasks.who_am_i, "abcdefgh"))))        # different worker pids

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[8, 9, 1000]
['a ran in process 6282', 'b ran in process 6283', 'c ran in process 6283', 'd ran in process 6283', 'e ran in process 6283', 'f ran in process 6283', 'g ran in process 6283', 'h ran in process 6282']


In [5]:
with mp.Pool(4) as pool:
    for result in pool.imap(mp_tasks.slow_square, range(5)):           # lazy, in order
        print("imap:", result, end="  ")
    print()
    print("imap_unordered:", list(pool.imap_unordered(mp_tasks.slow_square, range(5))))   # as they finish
    async_result = pool.apply_async(mp_tasks.power, (2, 100))                           # one task, non-blocking
    print("apply_async:", async_result.get(timeout=5))

imap: 0  imap: 1  imap: 4  imap: 9  imap: 16  


imap_unordered: [4, 1, 0, 9, 16]
apply_async: 1267650600228229401496703205376


### `chunksize` matters

Each task is pickled and sent through a pipe. For many tiny tasks, sending them in **chunks** reduces overhead dramatically:

In [6]:
nums = range(200_000)
for chunksize in (1, 1_000):
    start = time.perf_counter()
    with mp.Pool(4) as pool:
        n_primes = sum(pool.map(mp_tasks.is_prime, nums, chunksize=chunksize))
    print(f"chunksize={chunksize:<5} primes={n_primes} time={time.perf_counter() - start:.2f}s")

chunksize=1     primes=17984 time=2.31s


chunksize=1000  primes=17984 time=0.07s


### Per-worker initialisation

In [7]:
with mp.Pool(2, initializer=mp_tasks.init_worker, initargs=("job",)) as pool:   # e.g. open a DB connection once per worker
    print(pool.map(mp_tasks.tagged, [1, 2, 3]))

['job-1', 'job-2', 'job-3']


Other options: `maxtasksperchild=N` (recycle workers to contain memory leaks), `pool.close()` + `pool.join()` (graceful), `pool.terminate()` (immediate).

## 4. Communication between processes

### `Queue`

In [8]:
q, results = mp.Queue(), mp.Queue()
prod = mp.Process(target=mp_tasks.producer, args=(q, 5))
cons = mp.Process(target=mp_tasks.consumer, args=(q, results))
prod.start(); cons.start(); prod.join(); cons.join()
print([results.get() for _ in range(5)])

[0, 10, 20, 30, 40]


### `Pipe`: a two-way connection between exactly two endpoints

In [9]:
parent_conn, child_conn = mp.Pipe()
p = mp.Process(target=mp_tasks.ping_pong, args=(child_conn,))
p.start()
parent_conn.send("ping")
print(parent_conn.recv())
p.join()

pong (ping)


## 5. Sharing state

Processes have **separate memory**: a global variable changed in a child doesn't change in the parent. Options:

### `Value` and `Array` (shared ctypes memory)

In [10]:
counter = mp.Value("i", 0)            # "i" = C int
lock = mp.Lock()
procs = [mp.Process(target=mp_tasks.add_to_counter, args=(counter, lock, 1000)) for _ in range(4)]
for p in procs: p.start()
for p in procs: p.join()
print("counter:", counter.value)

arr = mp.Array("d", 5)                # 5 doubles
p = mp.Process(target=mp_tasks.fill_array, args=(arr, 100)); p.start(); p.join()
print(arr[:])

counter: 4000


[100.0, 101.0, 102.0, 103.0, 104.0]


`mp.Value` has its own lock too: `with counter.get_lock(): ...`.

### `shared_memory`: large blocks, zero-copy (great with NumPy)

In [11]:
from multiprocessing import shared_memory
import array

data = array.array("i", [1, 2, 3, 4, 5])
shm = shared_memory.SharedMemory(create=True, size=data.itemsize * len(data))
shm.buf[: len(data) * data.itemsize] = data.tobytes()
p = mp.Process(target=mp_tasks.double_shm, args=(shm.name, len(data))); p.start(); p.join()
print(array.array("i", bytes(shm.buf[: len(data) * data.itemsize])).tolist())
shm.close(); shm.unlink()            # always free shared memory

[2, 4, 6, 8, 10]


With NumPy: `np.ndarray(shape, dtype, buffer=shm.buf)` gives an array backed by the shared block.

### `Manager`: shared Python objects (dict, list, ...) via a server process

In [12]:
with mp.Manager() as manager:
    d, lst = manager.dict(), manager.list()
    procs = [mp.Process(target=mp_tasks.record, args=(d, lst, f"task{i}")) for i in range(3)]
    for p in procs: p.start()
    for p in procs: p.join()
    print(sorted(dict(d)), sorted(lst))

['task0', 'task1', 'task2'] ['task0', 'task1', 'task2']


Managers are flexible (and can work across machines) but much slower than `Value`/`shared_memory`, because every access is a message to the manager process.

## 6. Pickling constraints

Everything sent to a child (function, arguments, results) is **pickled**:
- ✅ module-level functions/classes, built-in types, dataclasses
- ❌ lambdas, nested/local functions, open files, sockets, locks (except mp's own), DB connections

In [13]:
try:
    with mp.Pool(2) as pool:
        pool.map(lambda x: x * 2, [1, 2])
except Exception as e:
    print(type(e).__name__, "→", str(e)[:80])

PicklingError → Can't pickle <function <lambda> at 0x107d2ada0>: it's not found as __main__.<lam


## 7. Tips
- Use processes for **CPU-bound** work; keep inputs/outputs small (pickling cost)
- Use `chunksize` for many small tasks; `imap_unordered` to process results as they arrive
- Prefer `concurrent.futures.ProcessPoolExecutor` (lesson 04) for a simpler, uniform API
- Consider `InterpreterPoolExecutor` (3.14) or the free-threaded build for lower overhead (lesson 08)
- On Linux in containers, `os.process_cpu_count()` (3.13) respects CPU affinity limits better than `os.cpu_count()`

In [14]:
print(os.cpu_count(), getattr(os, "process_cpu_count", lambda: "n/a")())

8 8


## 📝 Summary
- `Process` for individual workers; `Pool` (`map`/`imap`/`starmap`/`apply_async`, `chunksize`, initialisers) for batches.
- Start methods: `spawn` (macOS/Windows), `forkserver` (Linux 3.14+), `fork` (legacy, thread-unsafe).
- Communicate with `Queue`/`Pipe`; share state with `Value`/`Array`/`shared_memory`/`Manager`.
- Everything crossing process boundaries must be picklable; put workers in importable modules.

## 🏋️ Practice
1. Count primes below 2,000,000 with `Pool.map` and different `chunksize` values; plot or print the timings.
2. Build a producer/consumer pipeline with two consumer processes and sentinels.
3. Share a 1,000,000-element NumPy array with workers via `shared_memory` and have each worker square a slice.